# Análise de Acurácia — Projeto SESI
## Treinamento de Leitura | LMM + Gráfico por Grupo e Sessão

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("Bibliotecas carregadas.")

## Célula 2 — Configurações
> **Edite aqui** os caminhos, legendas, cores e sessões antes de rodar.

In [ ]:
# ── Caminho dos dados ──────────────────────────────────────────────────────
CAMINHO_DADOS = r'C:\Users\lucas\OneDrive\Documentos\Documents\1.DOUTORADO\SESI_01\PROC_DATA\aceletra_g_acelerado\sessao_stats.xlsx'

# ── Sessões a analisar ─────────────────────────────────────────────────────
SESSOES = list(range(0, 10))   # sessões 1 a 9

# ── Legendas do gráfico (edite livremente) ─────────────────────────────────
TITULO_GRAFICO  = 'Trajetória de Acurácia por Grupo — Sessões 1 a 9'
EIXO_X          = 'Sessão'
EIXO_Y          = 'Acurácia média (%) ± EPM'
LEGENDA_GRUPO_0 = 'Grupo Controle'
LEGENDA_GRUPO_1 = 'Grupo Intervenção'
COR_GRUPO_0     = '#2166AC'
COR_GRUPO_1     = '#D6604D'
TAMANHO_FONTE   = 13

# ── Controles de exibição ──────────────────────────────────────────────────
MOSTRAR_ANOTACAO_LMM = False    # False para ocultar anotação LMM no gráfico
MOSTRAR_LINHA_TENDENCIA = True  # False para ocultar linha tracejada

# ── Saídas ─────────────────────────────────────────────────────────────────
OUTPUT_EXCEL   = 'resultados_acuracia.xlsx'
OUTPUT_GRAFICO = 'grafico_acuracia.png'

print("Configurações definidas.")

## Célula 3 — Carregamento e cálculo de acurácia

In [ ]:
df = pd.read_excel(CAMINHO_DADOS)
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
print(f"Grupos: {df['grupo'].value_counts().to_dict()}")
print(f"Colunas disponíveis (amostra): {df.columns[:10].tolist()}")

# Calcula acurácia (%) por sessão
for s in SESSOES:
    total              = df[f'ACERTOSnum{s}'] + df[f'ERROSnum{s}']
    df[f'acuracia{s}'] = df[f'ACERTOSnum{s}'] / total * 100

print(f"\nAcurácia calculada para sessões: {SESSOES}")
print(df[[f'acuracia{s}' for s in SESSOES]].describe().round(2))

## Célula 4 — Formato longo

In [ ]:
rows = []
for s in SESSOES:
    for _, row in df.iterrows():
        rows.append({
            'subject'  : row['SUBJID'],
            'grupo'    : f"Grupo {int(row['grupo'])}",
            'session'  : s,
            'acuracia' : row[f'acuracia{s}'],
            'acertos'  : row[f'ACERTOSnum{s}'],
            'erros'    : row[f'ERROSnum{s}'],
        })

df_long = pd.DataFrame(rows)

# Mapeamento de legendas
label_map = {
    'Grupo 0': LEGENDA_GRUPO_0,
    'Grupo 1': LEGENDA_GRUPO_1,
}
df_long['grupo_label'] = df_long['grupo'].map(label_map)
palette = {
    LEGENDA_GRUPO_0: COR_GRUPO_0,
    LEGENDA_GRUPO_1: COR_GRUPO_1,
}

print(f"Formato longo: {df_long.shape}")
print(f"Sujeitos: {df_long['subject'].nunique()}")
print(f"Sessões:  {sorted(df_long['session'].unique())}")
print(f"Grupos:   {df_long['grupo_label'].unique()}")
print(df_long.head(4))

## Célula 5 — Estatísticas descritivas

In [ ]:
resumo = (
    df_long
    .groupby(['session', 'grupo_label'])['acuracia']
    .agg(
        n       = 'count',
        media   = 'mean',
        dp      = 'std',
        epm     = 'sem',
        mediana = 'median',
        minimo  = 'min',
        maximo  = 'max',
    )
    .round(2)
    .reset_index()
)

print("=== Estatísticas descritivas por sessão e grupo ===")
print(resumo.to_string(index=False))

## Célula 6 — LMM (Modelo Linear Misto)
Modelo: `acuracia ~ sessão × grupo + (1|sujeito)`

In [ ]:
print("=" * 55)
print("  LMM — Acurácia ~ sessão × grupo + (1|sujeito)")
print("=" * 55)

lmm_result = None

for metodo in ['bfgs', 'nm', 'lbfgs', 'powell']:
    try:
        lmm = smf.mixedlm(
            "acuracia ~ session * C(grupo, Treatment('Grupo 0'))",
            data   = df_long,
            groups = df_long["subject"]
        )
        lmm_result = lmm.fit(method=metodo, reml=True, disp=False)
        print(f"Método '{metodo}' convergiu.\n")
        break
    except Exception as e:
        print(f"Método '{metodo}' falhou: {type(e).__name__}")

if lmm_result is not None:
    print(lmm_result.summary())
    print("\n=== RESUMO DOS EFEITOS ===")
    params = lmm_result.params
    pvals  = lmm_result.pvalues
    for nome, beta, p in zip(params.index, params.values, pvals.values):
        sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
        print(f"  {nome:<52} β={beta:+.4f}  p={p:.4f}  {sig}")
else:
    print("Nenhum método convergiu.")

## Célula 7 — Gráfico
> Para abrir em **janela separada**: descomente `%matplotlib qt`  
> Para exibir **inline**: mantenha `%matplotlib inline` (padrão)

In [ ]:
# Descomente a linha desejada:
# %matplotlib qt       # janela separada (editável interativamente)
# matplotlib inline     # exibe no notebook

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor('white')

for grupo_label, cor in palette.items():
    sub = resumo[resumo['grupo_label'] == grupo_label].sort_values('session')
    y   = sub['media'].values
    ye  = sub['epm'].values
    x   = sub['session'].values

    # Banda de erro
    ax.fill_between(x, y - ye, y + ye, color=cor, alpha=0.15)

    # Linha de média
    ax.plot(x, y, 'o-', color=cor, linewidth=2.5,
            markersize=7, markerfacecolor=cor,
            markeredgecolor='white', markeredgewidth=1,
            label=grupo_label)

    # Linha de tendência
    if MOSTRAR_LINHA_TENDENCIA:
        m, b, *_ = stats.linregress(x, y)
        x_line   = np.linspace(x.min(), x.max(), 100)
        ax.plot(x_line, m*x_line + b, '--', color=cor,
                linewidth=1.2, alpha=0.6)

# Anotação LMM
if MOSTRAR_ANOTACAO_LMM and lmm_result is not None:
    try:
        chaves = [k for k in lmm_result.params.index
                  if 'session' in k and 'Grupo 1' in k]
        if chaves:
            k    = chaves[0]
            beta = lmm_result.params[k]
            p    = lmm_result.pvalues[k]
            sig  = '* p < 0.05' if p < 0.05 else 'n.s.'
            pstr = f'{p:.3f}' if p >= 0.001 else '< 0.001'
            ax.annotate(
                f'Sessão × Grupo: β = {beta:.4f}, p = {pstr} ({sig})',
                xy=(0.03, 0.05), xycoords='axes fraction',
                fontsize=10, color='#333333',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='lightyellow',
                          edgecolor='grey', alpha=0.85)
            )
    except Exception:
        pass

# Formatação
ax.set_title(TITULO_GRAFICO,
             fontsize=TAMANHO_FONTE + 1, fontweight='bold', pad=14)
ax.set_xlabel(EIXO_X,  fontsize=TAMANHO_FONTE)
ax.set_ylabel(EIXO_Y,  fontsize=TAMANHO_FONTE)
ax.set_xticks(SESSOES)
ax.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{x:.1f}%'))
ax.set_ylim(0,100)  # Limites fixos para melhor comparação entre grupos
    #max(0,   resumo['media'].min() - resumo['epm'].max() * 3),
    #min(100, resumo['media'].max() + resumo['epm'].max() * 3)

ax.legend(title='Grupo', fontsize=11, title_fontsize=11,
          frameon=True, framealpha=0.9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_GRAFICO, dpi=300, bbox_inches='tight', facecolor='white')
print(f"Figura salva: {OUTPUT_GRAFICO}")
plt.show()

## Célula 8 — Exporta Excel

In [ ]:
# Tabela LMM
if lmm_result is not None:
    fe_idx = lmm_result.fe_params.index
    ci     = lmm_result.conf_int().loc[fe_idx]
    df_lmm = pd.DataFrame({
        'Preditor'  : fe_idx,
        'β'         : lmm_result.fe_params.values,
        'Std.Err.'  : lmm_result.bse.loc[fe_idx].values,
        'z'         : (lmm_result.fe_params /
                       lmm_result.bse.loc[fe_idx]).values,
        'p-valor'   : lmm_result.pvalues.loc[fe_idx].values,
        'IC 2.5%'   : ci[0].values,
        'IC 97.5%'  : ci[1].values,
    }).round(4)
else:
    df_lmm = pd.DataFrame({'Aviso': ['Modelo não convergiu']})

# Formato wide
df_wide = df[['SUBJID','grupo'] +
             [f'acuracia{s}' for s in SESSOES]].copy()
df_wide.columns = (['subject','grupo'] +
                   [f'acuracia_s{s}' for s in SESSOES])

with pd.ExcelWriter(OUTPUT_EXCEL, engine='openpyxl') as w:
    df_lmm.to_excel(w,  sheet_name='LMM_Acuracia',  index=False)
    resumo.to_excel(w,  sheet_name='Descritivas',    index=False)
    df_long.to_excel(w, sheet_name='Formato_Longo',  index=False)
    df_wide.to_excel(w, sheet_name='Formato_Wide',   index=False)

print(f"Excel salvo: {OUTPUT_EXCEL}")
print("\n=== ANÁLISE CONCLUÍDA ===")